Dylan Ross

Compute associations between significantly altered lipids and proteins

## Setup

### Imports

In [1]:
import os

import networkx as nx
import numpy as np
import polars as pl
from scipy import stats
from matplotlib import pyplot as plt, rcParams
import synapseclient

# set max font size 
rcParams["font.size"] = 7

### Constants

In [ ]:
# Jupyter server should be running from src/python
# assumes auth_token.txt exists in src/python as described in setup instructions
AUTH_TOKEN = os.path.abspath("auth_token.txt")

# synapse IDs for the subset data
SUBSET_DATA_SYN_IDS = {
    "B_FLT3-ITD-WT.arrow": "syn74776638",
    "B_FLT3-ITD-WT.csv": "syn74776648",
    "B_FLT3-ITD-WT_mut-cols.txt": "syn74776635",
    "B_FLT3-ITD-WT_wt-cols.txt": "syn74776634",
    "B_FLT3-ITDxNPM1-WT.arrow": "syn74776644",
    "B_FLT3-ITDxNPM1-WT.csv": "syn74776636",
    "B_FLT3-ITDxNPM1-WT_mut-cols.txt": "syn74776633",
    "B_FLT3-ITDxNPM1-WT_wt-cols.txt": "syn74776632",
    "B_NPM1-WT.arrow": "syn74776645",
    "B_NPM1-WT.csv": "syn74776649",
    "B_NPM1-WT_mut-cols.txt": "syn74776642",
    "B_NPM1-WT_wt-cols.txt": "syn74776643",
    "W_FLT3-ITD-WT.arrow": "syn74776654",
    "W_FLT3-ITD-WT.csv": "syn74776655",
    "W_FLT3-ITD-WT_mut-cols.txt": "syn74776647",
    "W_FLT3-ITD-WT_wt-cols.txt": "syn74776646",
    "W_FLT3-ITDxNPM1-WT.arrow": "syn74776658",
    "W_FLT3-ITDxNPM1-WT.csv": "syn74776663",
    "W_FLT3-ITDxNPM1-WT_mut-cols.txt": "syn74776650",
    "W_FLT3-ITDxNPM1-WT_wt-cols.txt": "syn74776653",
    "W_NPM1-WT.arrow": "syn74776666",
    "W_NPM1-WT.csv": "syn74776667",
    "W_NPM1-WT_mut-cols.txt": "syn74776659",
    "W_NPM1-WT_wt-cols.txt": "syn74776660",
}

# cached table with top up and down features, apache arrow format, on synapse 
TOP_FEATURES_SYNID = "syn74782709"

CACHE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "mutations",
    "_cache"
)

# directory for figures
FIGURE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "mutations",
    "_figures",
    "assoc"
)

# directory for differentially expressed features
DIFFEX_DIR = os.path.join(
    CACHE_DIR,
    "diffex"
)

### Synapse login

In [3]:
with open(AUTH_TOKEN, "r") as atf:
    SYN = synapseclient.login(authToken=atf.read())


UPGRADE AVAILABLE

A more recent version of the Synapse Client (4.12.0) is available. Your version (4.11.0) can be upgraded by typing:
   pip install --upgrade synapseclient

Python Synapse Client version 4.12.0 release notes

https://python-docs.synapse.org/news/


Welcome, dylan.ross!



### Utility

In [4]:
def load_data(
    subset_id: str,
    block: str
) :
    """ 
    loads a specified -omics block for a selected data subset
    """
    # load subset sample columns
    with open(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}_wt-cols.txt"]).path, "r") as wtf:
        wt_cols = wtf.read().split("\n")
    with open(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}_mut-cols.txt"]).path, "r") as mutf:
        mut_cols = mutf.read().split("\n")

    # load subset omics data
    omics_subset = pl.read_ipc(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}.arrow"]).path)

    # select out the block of interest, drop null columns, return
    subset = (
        omics_subset
        .filter(pl.col("Block") == block)
        .drop("Block")
    )
    # drop any all-null columns
    return wt_cols, mut_cols, subset[[
        c.name 
        for c in subset 
        if not (c.null_count() == subset.height)
    ]]

In [5]:
def load_diffex_data_for_top_up_down_lipids(
    subset_id: str,
    up_or_down: str
) :
    # load the top up or down lipid features (from the table on synapse)
    top_up_down_lipids = (
        pl.read_ipc(SYN.get(TOP_FEATURES_SYNID).path)
        .filter(
            (pl.col("omics_type") == "Lipidomics")
            & (pl.col("population") == {"B": "Black", "W": "White"}[subset_id[0]])
            & (pl.col("mutation_comparison") == subset_id.split("_", 1)[1])
            & (
                (pl.col("delta_z") > 0) if up_or_down == "up" else (pl.col("delta_z") < 0)
            )
        )
    )["Feature"].to_list()
    # load the significant feature data
    subset = (
        # fetch the apache arrow table from synapse
        pl.read_ipc(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}.arrow"]).path)
        # select the specific block
        .filter(pl.col("Block") == "Lipidomics")
        # select only the top up/down lipids
        .filter(pl.col("Feature").is_in(top_up_down_lipids))
        # keep only the z-scores
        .drop(
            "Block",
        )
    )
    # drop any all-null columns
    return subset[[
        c.name 
        for c in subset 
        if not (c.null_count() == subset.height)
    ]]

In [6]:
def correlate_with_lipids(
    subset_id,
    block,
    up_or_down_lipids
) : 
    print("=" * 60)
    print(f"{subset_id=} {block=} {up_or_down_lipids=}")
    # load lipidomics and protein data, select common samples
    wt_cols, mut_cols, p_df = load_data(subset_id, block)
    l_df = load_diffex_data_for_top_up_down_lipids(
        subset_id, 
        up_or_down_lipids
    )
    common_cols = set(p_df.columns) & set(l_df.columns)
    print(f"{len(common_cols)} columns in common")
    print(f"\t{len(set(wt_cols) & common_cols)} WT samples")
    print(f"\t{len(set(mut_cols) & common_cols)} mutant samples")
    # check that there are sufficient numbers of WT and mutant samples
    # among the common columns
    p_mat = p_df.select(common_cols).drop("Feature").to_numpy()
    l_mat = l_df.select(common_cols).drop("Feature").to_numpy()
    # compute correlations with p-values
    results = []
    for p, p_data in zip(p_df["Feature"], p_mat):
        # fill NaNs with row minimum
        p_data[np.isnan(p_data)] = np.nanmin(p_data)
        for l, l_data in zip(l_df["Feature"], l_mat):
            # fill NaNs with row minimum
            l_data[np.isnan(l_data)] = np.nanmin(l_data)
            r, pv = stats.spearmanr(p_data, l_data)
            results.append({
                "protein": p,
                "lipid": l,
                "correlation": r,
                "p_value": pv,
                "p_data": p_data,
                "l_data": l_data
            })
    if len(results) < 1:
        return None
    # create a dataframe from the correlation results
    corr = (
        pl.DataFrame(results)
        .sort(
            pl.col("correlation").abs(),
            descending=True
        )
        .filter(
            pl.col("p_value").is_not_nan()
        )
    )
    corr = (
        corr
        # correct p-values for multiple comparisons
        # Benjamini-Hochberg
        .with_columns(
            pl.Series(
                "p_adj", 
                stats.false_discovery_control(corr["p_value"].to_numpy())
            )
        )
        # only keep correlations that are significant
        # and have a moderate effect size
        .filter(
            (pl.col("p_adj") <= 0.05)
            & (pl.col("correlation") >= 0.5)
        )
    )
    print(f"{len(corr)} correlations meet significant and correlation threshold")
    if len(corr) > 0:
        print(f"\t{len(corr["protein"].unique())} proteins")
        print(f"\t{len(corr["lipid"].unique())} lipids")
        return corr
    return None

In [10]:
def write_associated_protein_list(
    corrs,
    subset,
    up_or_down_lipids
):
    corr_tx = corrs[(subset, "Transcriptomics", up_or_down_lipids)]
    corr_px = corrs[(subset, "Proteomics", up_or_down_lipids)]
    df = None
    if corr_tx is not None and corr_px is not None:
        df = pl.concat([
            corr_tx,
            corr_px
        ])
    elif corr_tx is not None:
        df = corr_tx
    elif corr_px is not None:
        df = corr_px
    if df is not None:
        (
            df    
            .group_by("protein")
            .agg(
                "lipid",
                "correlation",
                "p_adj",
                "p_data",
                "l_data"
            )
            # sort according to the median adjusted p-value for any
            # significant correlation between this protein and lipids
            # the top proteins should have strong correlations between
            # ideally multiple individual lipid species
            .sort(pl.col("p_adj").list.min())
            .head(250)
            .select("protein")
            .write_csv(
                os.path.join(
                    CACHE_DIR,
                    "diffex",
                    f"{subset}_Lipid-Protein-Associations_{up_or_down_lipids}.txt"
                ),
                include_header=False
            )
        )

## Analysis

In [8]:
corrs = {
    (subset_id, block, up_or_down_lipids): correlate_with_lipids(
        subset_id,
        block,
        up_or_down_lipids
    )
    for subset_id in [
        "B_NPM1-WT", 
        "W_NPM1-WT", 
        "B_FLT3-ITD-WT", 
        "W_FLT3-ITD-WT", 
        "B_FLT3-ITDxNPM1-WT", 
        "W_FLT3-ITDxNPM1-WT"
    ]
    for block in [
        "Transcriptomics", 
        "Proteomics", 
    ]
    for up_or_down_lipids in [
        "up",
        "down"
    ]
}

[WARNING] /var/folders/nb/ykfs58f552d0d0jd_2jnkx_80000gp/T/ipykernel_32084/3570952081.py:9: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  with open(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}_wt-cols.txt"]).path, "r") as wtf:



subset_id='B_NPM1-WT' block='Transcriptomics' up_or_down_lipids='up'
[syn74776643:B_NPM1-WT_wt-cols.txt]: Found existing file at /Users/dylan.ross/.synapseCache/659/172720659/B_NPM1-WT_wt-cols.txt, skipping download.


[WARNING] /var/folders/nb/ykfs58f552d0d0jd_2jnkx_80000gp/T/ipykernel_32084/3570952081.py:11: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  with open(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}_mut-cols.txt"]).path, "r") as mutf:



[syn74776642:B_NPM1-WT_mut-cols.txt]: Found existing file at /Users/dylan.ross/.synapseCache/658/172720658/B_NPM1-WT_mut-cols.txt, skipping download.


[WARNING] /var/folders/nb/ykfs58f552d0d0jd_2jnkx_80000gp/T/ipykernel_32084/3570952081.py:15: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  omics_subset = pl.read_ipc(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}.arrow"]).path)



[syn74776645:B_NPM1-WT.arrow]: Found existing file at /Users/dylan.ross/.synapseCache/663/172720663/B_NPM1-WT.arrow, skipping download.


[WARNING] /var/folders/nb/ykfs58f552d0d0jd_2jnkx_80000gp/T/ipykernel_32084/2368620118.py:7: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  pl.read_ipc(SYN.get(TOP_FEATURES_SYNID).path)



[syn74782709:all_top_features.arrow]: Found existing file at /Users/dylan.ross/.synapseCache/365/172730365/all_top_features.arrow, skipping download.


[WARNING] /var/folders/nb/ykfs58f552d0d0jd_2jnkx_80000gp/T/ipykernel_32084/2368620118.py:20: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  pl.read_ipc(SYN.get(SUBSET_DATA_SYN_IDS[f"{subset_id}.arrow"]).path)



[syn74776645:B_NPM1-WT.arrow]: Found existing file at /Users/dylan.ross/.synapseCache/663/172720663/B_NPM1-WT.arrow, skipping download.
54 columns in common
	47 WT samples
	6 mutant samples
489 correlations meet significant and correlation threshold
	353 proteins
	22 lipids
subset_id='B_NPM1-WT' block='Transcriptomics' up_or_down_lipids='down'
[syn74776643:B_NPM1-WT_wt-cols.txt]: Found existing file at /Users/dylan.ross/.synapseCache/659/172720659/B_NPM1-WT_wt-cols.txt, skipping download.
[syn74776642:B_NPM1-WT_mut-cols.txt]: Found existing file at /Users/dylan.ross/.synapseCache/658/172720658/B_NPM1-WT_mut-cols.txt, skipping download.
[syn74776645:B_NPM1-WT.arrow]: Found existing file at /Users/dylan.ross/.synapseCache/663/172720663/B_NPM1-WT.arrow, skipping download.
[syn74782709:all_top_features.arrow]: Found existing file at /Users/dylan.ross/.synapseCache/365/172730365/all_top_features.arrow, skipping download.
[syn74776645:B_NPM1-WT.arrow]: Found existing file at /Users/dylan.ros

In [11]:
for subset_id in [
    "B_NPM1-WT", 
    "W_NPM1-WT", 
    "B_FLT3-ITD-WT", 
    "W_FLT3-ITD-WT", 
    "B_FLT3-ITDxNPM1-WT", 
    "W_FLT3-ITDxNPM1-WT"
]:
    for up_or_down_lipids in [
        "up",
        "down"
    ]:
        write_associated_protein_list(
            corrs,
            subset_id,
            up_or_down_lipids
        )

In [ ]:
# combine all protein-lipid associations into a table

In [37]:
all_lipid_protein_correlations = (
    pl.concat([
        (
           df
           .drop(
               "p_data",
               "l_data",
               "p_value"
           )
           .rename({
               "correlation": "spearman_r"
           })
           .with_columns(
               pl.lit({"B": "Black", "W": "White"}[subset_id[0]]).alias("population"),
               pl.lit(subset_id.split("_", 1)[1]).alias("mutation_comparison"),
               pl.lit(direction).alias("lipid_direction"),
               pl.lit(omics_type).alias("protein_omics_type")
           )
        )
        for (subset_id, omics_type, direction), df in corrs.items()
        if df is not None
    ])
)
all_lipid_protein_correlations

protein,lipid,spearman_r,p_adj,population,mutation_comparison,lipid_direction,protein_omics_type
str,str,f64,f64,str,str,str,str
"""PAQR3""","""neg_PE 36:5|PE 16:1_20:4_[M-H]…",0.685131,0.00775,"""Black""","""NPM1-WT""","""up""","""Transcriptomics"""
"""ACTR5""","""pos_PE P-38:4|PE P-18:0_20:4_[…",0.656426,0.01266,"""Black""","""NPM1-WT""","""up""","""Transcriptomics"""
"""TRIM37""","""neg_PE 38:5|PE 18:1_20:4_[M-H]…",0.64667,0.01266,"""Black""","""NPM1-WT""","""up""","""Transcriptomics"""
"""TRIM37""","""neg_PE 36:5|PE 16:1_20:4_[M-H]…",0.645622,0.01266,"""Black""","""NPM1-WT""","""up""","""Transcriptomics"""
"""ENSG00000223459""","""pos_PC 38:5|PC 18:1_20:4_[M+H]…",0.638204,0.01266,"""Black""","""NPM1-WT""","""up""","""Transcriptomics"""
…,…,…,…,…,…,…,…
"""TLK1""","""neg_Cer 34:2;O2|Cer 18:2;O2/16…",0.50011,0.000868,"""White""","""FLT3-ITDxNPM1-WT""","""up""","""Proteomics"""
"""KDM2A""","""neg_PG 36:4|PG 16:0_20:4_[M-H]…",0.500073,0.000869,"""White""","""FLT3-ITDxNPM1-WT""","""up""","""Proteomics"""
"""B3GALT6""","""neg_PG 36:4|PG 16:0_20:4_[M-H]…",0.500037,0.00087,"""White""","""FLT3-ITDxNPM1-WT""","""up""","""Proteomics"""


In [38]:
all_lipid_protein_correlations.write_ipc(
    os.path.join(
        CACHE_DIR, 
        "all_lipid-protein_correlations.arrow"
    )
)

In [39]:
all_lipid_protein_correlations.write_csv(
    os.path.join(
        CACHE_DIR, 
        "all_lipid-protein_correlations.csv"
    )
)

# TODO
Automate uploading/updating the all lipid-protein correlations table (apache arrow + csv) to synapse (`syn74802631` and `syn74802632`, respectively). In the meantime, I manually uploaded them.